# Qwen Model Quantization on Google Colab

This notebook quantizes Qwen 3.5/2.5 models to Q4_K_M format for mobile deployment.

**Total Time**: ~15-20 minutes  
**Output Size**: ~2.0 GB (from 7.3 GB input)  
**Compression**: 3.6x smaller

---

## Step 1: Install Dependencies

In [ ]:
!pip install -q llama-cpp-python --prefer-binary
!pip install -q huggingface-hub tqdm requests
!apt-get install -y -qq cmake ninja-build git

print("✅ Dependencies installed")

## Step 2: Check Colab Environment

In [ ]:
import psutil
import subprocess

print("\n" + "="*70)
print("COLAB ENVIRONMENT INFO")
print("="*70 + "\n")

# Check GPU
gpu_check = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], 
                          capture_output=True, text=True, stderr=subprocess.DEVNULL)
if gpu_check.returncode == 0:
    print(f"GPU: {gpu_check.stdout.strip()}")
else:
    print("GPU: None (CPU only)")

# Check RAM
total_ram = psutil.virtual_memory().total / (1024**3)
print(f"RAM: {total_ram:.1f} GB")

# Check Storage
total_disk = psutil.disk_usage('/').total / (1024**3)
free_disk = psutil.disk_usage('/').free / (1024**3)
print(f"Storage: {total_disk:.1f} GB total, {free_disk:.1f} GB available")

print("="*70)
print("\n✅ Environment check complete")

## Step 3: Download Qwen Model from HuggingFace

This downloads the base Q8_0 model (~7.3 GB)  
**Time**: ~5-10 minutes depending on connection speed

In [ ]:
from huggingface_hub import hf_hub_download
import os
from pathlib import Path

print("\n" + "="*70)
print("DOWNLOADING QWEN MODEL")
print("="*70 + "\n")

# Configuration
REPO_ID = "Qwen/Qwen2.5-7B-Instruct-GGUF"  # Qwen 2.5 7B
MODEL_FILE = "qwen2.5-7b-instruct-q8_0.gguf"  # Base high-precision

print(f"Repository: {REPO_ID}")
print(f"Model File: {MODEL_FILE}")
print(f"Download Size: ~7.3 GB")
print()

model_path = hf_hub_download(
    repo_id=REPO_ID,
    filename=MODEL_FILE,
    local_dir="./models",
    local_dir_use_symlinks=False,
)

print(f"\n✅ Model downloaded!")
print(f"   Path: {model_path}")
print(f"   Size: {os.path.getsize(model_path) / (1024**3):.2f} GB")

# Store path for next cell
input_model_path = model_path

## Step 4: Quantize to Q4_K_M Format

Quantization process:  
- Input: 7.3 GB (Q8_0)
- Output: 2.0 GB (Q4_K_M)
- Compression: 3.6x
- **Time**: ~10 minutes

In [ ]:
import llama_cpp
import time
import os
from pathlib import Path

print("\n" + "="*70)
print("QUANTIZING MODEL TO Q4_K_M")
print("="*70 + "\n")

# Paths
input_model = "./models/qwen2.5-7b-instruct-q8_0.gguf"
output_model = "./models/qwen2.5-7b-instruct-q4_k_m.gguf"

input_size = os.path.getsize(input_model)
print(f"Input Model:  {Path(input_model).name}")
print(f"Input Size:   {input_size / (1024**3):.2f} GB")
print(f"Output Model: {Path(output_model).name}")
print(f"Format:       Q4_K_M (4-bit, balanced)")
print(f"\nStarting quantization...")
print("-" * 70)

start_time = time.time()

# Quantization type Q4_K_M = 25 in llama_cpp
llama_cpp.llama_model_quantize(
    input_model.encode("utf-8"),
    output_model.encode("utf-8"),
    llama_cpp.llama_model_quantize_params(qtype=25)  # Q4_K_M
)

elapsed = time.time() - start_time

if os.path.exists(output_model):
    output_size = os.path.getsize(output_model)
    ratio = input_size / output_size
    
    print(f"\n{'='*70}")
    print(f"✅ QUANTIZATION COMPLETE!")
    print(f"{'='*70}")
    print(f"Output Size:  {output_size / (1024**3):.2f} GB")
    print(f"Compression: {ratio:.1f}x")
    print(f"Space Saved: {(input_size - output_size) / (1024**3):.2f} GB")
    print(f"Time:        {elapsed / 60:.1f} minutes")
    print(f"Speed:       {input_size / elapsed / (1024**3):.2f} GB/sec")
    print(f"{'='*70}\n")
else:
    print("\n❌ Quantization failed!")

# Store output path
quantized_model_path = output_model

## Step 5: Download Quantized Model

In [ ]:
from google.colab import files
import os

print("\n" + "="*70)
print("DOWNLOADING QUANTIZED MODEL")
print("="*70 + "\n")

model_file = "./models/qwen2.5-7b-instruct-q4_k_m.gguf"

if os.path.exists(model_file):
    file_size = os.path.getsize(model_file) / (1024**3)
    print(f"File: qwen2.5-7b-instruct-q4_k_m.gguf")
    print(f"Size: {file_size:.2f} GB")
    print(f"\nClick 'Save' when prompted in the download dialog.\n")
    
    files.download(model_file)
    print("\n✅ Download completed!")
    print(f"   File saved to: ~/Downloads/qwen2.5-7b-instruct-q4_k_m.gguf")
else:
    print("❌ Model file not found. Run quantization cell first.")

## (Optional) Step 6: Quantize to Multiple Formats

Generate Q3_K_M, Q5_K_M, Q6_K formats for comparison

In [ ]:
# OPTIONAL: Quantize to multiple formats

import llama_cpp
import time
import os

formats = {
    "q3_k_m": 26,  # Low quality, maximum compression
    "q5_k_m": 28,  # High quality
    "q6_k": 27,    # Very high quality
}

input_model = "./models/qwen2.5-7b-instruct-q8_0.gguf"

print("\n" + "="*70)
print("QUANTIZING TO MULTIPLE FORMATS")
print("="*70 + "\n")

for fmt_name, fmt_id in formats.items():
    output_model = f"./models/qwen2.5-7b-instruct-{fmt_name}.gguf"
    
    print(f"Starting {fmt_name.upper()}...")
    start = time.time()
    
    llama_cpp.llama_model_quantize(
        input_model.encode("utf-8"),
        output_model.encode("utf-8"),
        llama_cpp.llama_model_quantize_params(qtype=fmt_id)
    )
    
    if os.path.exists(output_model):
        size_gb = os.path.getsize(output_model) / (1024**3)
        elapsed = time.time() - start
        print(f"  ✅ Size: {size_gb:.2f} GB  Time: {elapsed/60:.1f}m\n")
    else:
        print(f"  ❌ Failed\n")

print("="*70)
print("✅ All formats complete")

## (Optional) Step 7: List All Generated Models

In [ ]:
import os

print("\n" + "="*70)
print("AVAILABLE MODELS")
print("="*70 + "\n")

model_dir = "./models"
if os.path.exists(model_dir):
    models = [f for f in os.listdir(model_dir) if f.endswith('.gguf')]
    
    if models:
        print(f"{'Filename':<50} {'Size':<15}")
        print("-" * 65)
        
        for model in sorted(models):
            path = os.path.join(model_dir, model)
            size_gb = os.path.getsize(path) / (1024**3)
            print(f"{model:<50} {size_gb:>6.2f} GB")
        
        print("="*65)
    else:
        print("No models found.")
else:
    print("Models directory not found.")

## (Optional) Step 8: Save to Google Drive

Save quantized models to Google Drive for persistent storage

In [ ]:
from google.colab import drive
import shutil
import os

print("\nMounting Google Drive...")
drive.mount('/content/drive')

# Copy quantized model to Drive
model_file = "./models/qwen2.5-7b-instruct-q4_k_m.gguf"
drive_path = "/content/drive/MyDrive/orag_models/qwen2.5-7b-instruct-q4_k_m.gguf"

if os.path.exists(model_file):
    os.makedirs(os.path.dirname(drive_path), exist_ok=True)
    print(f"\nCopying to Google Drive...")
    shutil.copy(model_file, drive_path)
    print(f"✅ Saved to: {drive_path}")
    print(f"   Ready for future use across notebooks")
else:
    print("Model file not found.")

---

## Summary

✅ **Quantization Complete!**

### Output File
- **Name**: `qwen2.5-7b-instruct-q4_k_m.gguf`
- **Size**: ~2.0 GB
- **Quality**: Good for RAG + Chat
- **Compression**: 3.6x smaller

### Next Steps
1. Download the model from Step 5
2. Place in O-RAG project: `cp qwen2.5-7b-instruct-q4_k_m.gguf ~/Desktop/O-rag/`
3. Run O-RAG: `python main.py`
4. Models auto-load from local directory

### Use Cases
- **Mobile (4GB RAM)**: Q4_K_M ✅ Recommended
- **Desktop (8GB+ RAM)**: Q5_K_M or Q6_K
- **Extreme Compression**: Q3_K_M

---